# Colab SlideDeck 预处理实验

这个 Notebook 用 Colab GPU 验证单个 PDF/PPTX 的统一页级解析、资源筛选和切块，并下载完整预处理包。当前不会连接本地 Qdrant，也不会调用 Embedding API。

在 VS Code 右上角点击 **选择内核**，选择 **Colab → New Colab Server**，并为运行时选择 GPU。连接后从上到下执行。


In [ ]:
import os
import platform
import shutil
import subprocess
import sys

import torch

in_colab = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

print("运行环境:", "Google Colab" if in_colab else "本地或其他 Jupyter")
print("Python:", sys.version.split()[0])
print("系统:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA 编译版本:", torch.version.cuda)
print("CUDA 可用:", cuda_available)
print("Apple MPS 可用:", mps_available)

if cuda_available:
    print("GPU:", torch.cuda.get_device_name(0))
elif in_colab:
    print("\n当前 Colab 会话没有分配 GPU。请新建或切换到 GPU 运行时，然后重新连接内核。")
else:
    print("\n当前单元格没有在 Colab GPU 运行时执行。请在 VS Code 的内核选择器中连接 Colab GPU。")

## 1. 获取项目并安装文档解析依赖

Colab 是独立的远程文件系统，因此需要在运行时内克隆项目。运行时重启后通常需要重新执行此步骤。

In [ ]:
from getpass import getpass
from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile

ARCHIVE_URL = "https://api.github.com/repos/Drchennnn/SRAG/zipball/main"
ARCHIVE_PATH = Path("/content/SRAG-slidedeck.zip")
REPO_DIR = Path("/content/SRAG-slidedeck")
EXTRACT_DIR = Path("/content/SRAG-extracted")

if not (REPO_DIR / "pyproject.toml").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True)

    github_token = os.environ.get("GITHUB_TOKEN")
    if not github_token:
        try:
            from google.colab import userdata

            github_token = userdata.get("GITHUB_TOKEN")
        except Exception:
            github_token = None
    if not github_token:
        github_token = getpass("请输入 GitHub 只读 Token（输入不会显示）：").strip()
    if not github_token:
        raise RuntimeError("私有仓库需要 GITHUB_TOKEN。")

    print("正在下载 SRAG main 分支……")
    request = Request(
        ARCHIVE_URL,
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {github_token}",
            "User-Agent": "SRAG-Colab",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    with urlopen(request, timeout=60) as response, ARCHIVE_PATH.open("wb") as target:
        shutil.copyfileobj(response, target)
    del github_token

    with ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(EXTRACT_DIR)
    extracted_projects = [path for path in EXTRACT_DIR.iterdir() if (path / "pyproject.toml").exists()]
    if len(extracted_projects) != 1:
        raise RuntimeError("无法在下载内容中确定 SRAG 项目目录。")
    shutil.move(str(extracted_projects[0]), REPO_DIR)

if not (REPO_DIR / "pyproject.toml").exists():
    raise RuntimeError(f"项目下载不完整：{REPO_DIR}")

# 开发期间可把本地最新代码打包为 /content/srag_slide_code.zip，覆盖远端 main 中的旧文件。
LOCAL_CODE_ARCHIVE = Path("/content/srag_slide_code.zip")
if LOCAL_CODE_ARCHIVE.exists():
    print("正在应用本地 SlideDeck 开发代码……")
    with ZipFile(LOCAL_CODE_ARCHIVE) as archive:
        archive.extractall(REPO_DIR)

def run_checked(command: list[str]) -> None:
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"命令执行失败（退出码 {result.returncode}）：{' '.join(command)}")

print("正在安装 SRAG 和文档解析依赖……")
run_checked(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[documents]"]
)

os.environ["DOCLING_DEVICE"] = "cuda" if cuda_available else "cpu"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("项目目录:", REPO_DIR)
print("Docling 设备:", os.environ["DOCLING_DEVICE"])


## 2. 准备一个测试文件

先只测试一个 `.pdf` 或 `.pptx` 文件。可以在 VS Code 的 Colab 文件面板中上传到 `/content`，也可以挂载 Google Drive 后填写 Drive 路径。

上传后把 `FILE_PATH` 改成远程文件的绝对路径，例如 `/content/Lecture2a.pdf`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 填写 Colab 远程文件系统中的路径。
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/CS5187_Vision_and_Image/Week01/Lecture1-1.pdf"
# 只对原生文本很少的 PDF 页面启动 OCR；PPTX 不使用此开关。
ENABLE_PDF_OCR_FALLBACK = True

input_file = Path(FILE_PATH).expanduser() if FILE_PATH.strip() else None
if input_file is None:
    print("请先上传一个 PDF/PPTX，并填写 FILE_PATH。")
elif not input_file.exists():
    raise FileNotFoundError(f"文件不存在：{input_file}")
elif input_file.suffix.lower() not in {".pdf", ".pptx"}:
    raise ValueError("当前实验仅支持 .pdf 和 .pptx。")
else:
    print(f"输入文件：{input_file} ({input_file.stat().st_size / 1024 / 1024:.2f} MB)")
    print("PDF 按页 OCR 回退：", ENABLE_PDF_OCR_FALLBACK)

if input_file and input_file.suffix.lower() == ".pptx" and not shutil.which("libreoffice"):
    print("正在为 PPTX 整页截图安装 LibreOffice Impress……")
    run_checked(["apt-get", "update", "-qq"])
    run_checked(["apt-get", "install", "-y", "-qq", "libreoffice-impress"])


## 3. 解析、筛选资源并按页切块

PDF 优先读取原生文本，只对稀疏页回退 OCR；PPTX 读取文本框、表格、图表、图片和布局。两种格式都输出统一的 `SlideDeck`，一页/一张作为基础 Chunk，仅拆分超长页。


In [ ]:
import time

from app.rag.slide_chunker import SlideChunker
from app.rag.slide_preprocessor import SlideDeckPreprocessor

if input_file is None:
    raise ValueError("请先在上一个单元格填写 FILE_PATH。")

OUTPUT_ROOT = Path("/content/srag_outputs")
preprocessor = SlideDeckPreprocessor(
    output_root=OUTPUT_ROOT,
    chunker=SlideChunker(max_tokens=650, overlap_tokens=50),
    enable_pdf_ocr_fallback=ENABLE_PDF_OCR_FALLBACK,
    pdf_ocr_min_text_tokens=8,
)
started_at = time.perf_counter()
result = preprocessor.preprocess(input_file)
elapsed = time.perf_counter() - started_at

deck = result.deck
chunks = result.chunking.chunks
token_counts = [chunk.token_count for chunk in chunks]
asset_count = sum(len(slide.assets) for slide in deck.slides)
visual_count = sum(slide.visual_embedding_candidate for slide in deck.slides)
print(f"预处理完成：{elapsed:.1f} 秒")
print("文档 ID：", deck.document_id)
print("Slides：", len(deck.slides))
print("Assets：", asset_count)
print("视觉候选页：", visual_count)
print("Chunks：", len(chunks))
if token_counts:
    print("Token 数量（最小/平均/最大）：", min(token_counts), f"{sum(token_counts) / len(token_counts):.1f}", max(token_counts))

for index, chunk in enumerate(chunks[:3], start=1):
    print(f"\n--- Chunk {index} | 第 {chunk.slide_no} 页 | {chunk.token_count} tokens ---")
    print("标题：", chunk.title or "无")
    print(chunk.display_text[:800])


## 4. 打包并下载

导出目录包含 `deck.json`、`chunks.json`、`slides/` 和 `assets/`。下载 ZIP 后可在本地检查，下一阶段再接入 Embedding 与索引。


In [ ]:
import shutil

archive_base = Path("/content") / f"{input_file.stem}.slidedeck"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", result.bundle_dir))
print("预处理目录：", result.bundle_dir)
print(f"压缩包：{archive_path} ({archive_path.stat().st_size / 1024 / 1024:.2f} MB)")

from google.colab import files

files.download(str(archive_path))
